# Experiment 3 — Linear Absorption Coefficient (µ) of Radiation
### Tri-Chandra Multiple Campus | Department of Physics | B.Sc. III Year | 2082

**Instrument:** Geiger-Müller (GM) Counter  
**Objective:** Determine the linear absorption coefficient µ of radiation through a material and draw the best-fit line using the method of least squares.

---

## 📐 Theory

When ionising radiation passes through an absorber of thickness $x$, the transmitted count rate $N(x)$ follows the **Beer-Lambert exponential attenuation law**:

$$N(x) = N_0 \cdot e^{-\mu x}$$

where:
- $N_0$ = count rate without absorber (at $x = 0$)
- $\mu$ = **linear attenuation coefficient** (cm⁻¹)
- $x$ = absorber thickness (cm)

Taking the natural logarithm **linearises** the equation:

$$\ln N(x) = \ln N_0 - \mu \cdot x$$

This is of the form $y = mx + c$, so a plot of $\ln N$ vs $x$ yields:
- **Slope** $= -\mu$
- **Intercept** $= \ln N_0$

### Statistical Framework

GM counter measurements follow **Poisson statistics**: for $N$ counts, $\sigma_N = \sqrt{N}$.  
By error propagation:

$$\sigma_{\ln N} = \frac{\sigma_N}{N} = \frac{1}{\sqrt{N}}$$

Since uncertainties vary across data points, we use **Weighted Least Squares (WLS)** with weights $w_i = 1/\sigma_i^2$.

---
## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import linregress
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 130

---
## 2. Data

Counts recorded at each absorber thickness. Error = $\sqrt{N}$ (Poisson).

In [ ]:
raw = {
    'thickness_mm': [0,    2,    4,    6,    8,   10,   12,   14,   16,  18,  20],
    'counts'      : [1053, 768,  551,  412,  305,  218,  165,  121,   94,  66,  48],
    'count_error' : [32.45, 27.71, 23.47, 20.30, 17.46,
                     14.76, 12.85, 11.00,  9.70,  8.12, 6.93],
}

df = pd.DataFrame(raw)
df['thickness_cm'] = df['thickness_mm'] / 10.0        # convert to cm
df['ln_counts']    = np.log(df['counts'])              # linearise
df['ln_error']     = df['count_error'] / df['counts']  # propagated σ(lnN)

df.style.format({
    'thickness_mm': '{:.1f}',
    'thickness_cm': '{:.2f}',
    'counts': '{:.0f}',
    'count_error': '{:.2f}',
    'ln_counts': '{:.4f}',
    'ln_error': '{:.4f}',
}).set_caption('Observed GM Counter Readings with Poisson Errors')

---
## 3. Weighted Least Squares Fit

We minimise $\chi^2 = \sum_i w_i \left(y_i - a x_i - b\right)^2$ where $w_i = 1/\sigma_i^2$.

The closed-form solution:
$$a = \frac{S_w S_{wxy} - S_{wx} S_{wy}}{D}, \quad b = \frac{S_{wxx} S_{wy} - S_{wx} S_{wxy}}{D}$$
$$D = S_w S_{wxx} - S_{wx}^2$$
$$\sigma_a = \sqrt{S_w / D}, \quad \sigma_b = \sqrt{S_{wxx} / D}$$

In [ ]:
x    = df['thickness_cm'].values
y    = df['ln_counts'].values
yerr = df['ln_error'].values

# Weights
w    = 1.0 / yerr**2
Sw   = np.sum(w);           Swx  = np.sum(w * x)
Swy  = np.sum(w * y);       Swxx = np.sum(w * x**2)
Swxy = np.sum(w * x * y)

D    = Sw * Swxx - Swx**2

a_w  = (Sw * Swxy  - Swx * Swy)  / D     # slope  = -µ
b_w  = (Swxx * Swy - Swx * Swxy) / D     # intercept = ln N₀

da_w = np.sqrt(Sw   / D)                 # σ(slope)
db_w = np.sqrt(Swxx / D)                 # σ(intercept)

mu     = -a_w                            # attenuation coefficient
mu_err =  da_w
N0     =  np.exp(b_w)
N0_err =  N0 * db_w                      # propagated error

# Goodness of fit
_, _, r_value, p_value, _ = linregress(x, y)
y_pred    = a_w * x + b_w
residuals = y - y_pred
chi2_val  = np.sum((residuals / yerr)**2)
dof       = len(x) - 2
chi2_red  = chi2_val / dof

print("╔══════════════════════════════════════════════════════╗")
print("║          RESULTS — WEIGHTED LEAST SQUARES FIT        ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  µ  (attenuation coeff.)  = {mu:.4f} ± {mu_err:.4f} cm⁻¹   ║")
print(f"║  N₀ (initial count rate)  = {N0:.2f} ± {N0_err:.2f}       ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  R²           = {r_value**2:.6f}                         ║")
print(f"║  χ²_reduced   = {chi2_red:.4f}  (dof = {dof})                  ║")
print(f"║  p-value      = {p_value:.2e}                       ║")
print("╚══════════════════════════════════════════════════════╝")

---
## 4. Results Table

In [ ]:
df['ln_fit']   = a_w * x + b_w
df['residual'] = residuals

df[['thickness_mm','counts','count_error','ln_counts','ln_error','ln_fit','residual']]\
  .rename(columns={
      'thickness_mm':'x (mm)', 'counts':'N', 'count_error':'σ_N',
      'ln_counts':'ln(N)', 'ln_error':'σ_lnN', 'ln_fit':'ln(fit)', 'residual':'Δln(N)'
  })\
  .style.format({
      'x (mm)':'{:.1f}', 'N':'{:.0f}', 'σ_N':'{:.2f}',
      'ln(N)':'{:.4f}', 'σ_lnN':'{:.4f}', 'ln(fit)':'{:.4f}', 'Δln(N)':'{:.4f}'
  })\
  .background_gradient(subset=['Δln(N)'], cmap='RdYlGn_r')\
  .set_caption('Observed, Fitted, and Residual Values')

---
## 5. Visualisation

In [ ]:
x_fit = np.linspace(0, 2.1, 400)
y_fit = a_w * x_fit + b_w
N_fit = np.exp(b_w) * np.exp(a_w * x_fit)

BG = '#0d1117'; PANEL = '#161b22'; BORDER = '#30363d'; GRID = '#21262d'
BLUE = '#58a6ff'; ORANGE = '#f0883e'; GREEN = '#56d364'
PURPLE = '#bc8cff'; MUTED = '#8b949e'; WHITE = '#e6edf3'

fig = plt.figure(figsize=(14, 9), facecolor=BG)
gs  = gridspec.GridSpec(2, 2, figure=fig,
                        hspace=0.45, wspace=0.38,
                        left=0.09, right=0.95, top=0.91, bottom=0.09)

def style_ax(ax, title=''):
    ax.set_facecolor(PANEL)
    for sp in ax.spines.values(): sp.set_edgecolor(BORDER)
    ax.tick_params(colors=MUTED, labelsize=9)
    ax.xaxis.label.set_color(MUTED); ax.yaxis.label.set_color(MUTED)
    ax.grid(True, color=GRID, linewidth=0.7, linestyle='--', alpha=0.8)
    if title: ax.set_title(title, color=WHITE, fontsize=11, fontweight='bold', pad=8)

# Panel 1: ln(N) vs x — main plot
ax1 = fig.add_subplot(gs[0, :])
ax1.errorbar(x, y, yerr=yerr, fmt='o', color=BLUE, ecolor=GREEN,
             capsize=4, capthick=1.3, markersize=7, linewidth=0,
             label='ln(N)  with propagated errors', zorder=5)
ax1.plot(x_fit, y_fit, color=ORANGE, linewidth=2.3,
         label=f'WLS best fit: y = ({a_w:.4f}±{da_w:.4f})x + {b_w:.4f}', zorder=4)
ax1.fill_between(x_fit,
                 (a_w-da_w)*x_fit + (b_w-db_w),
                 (a_w+da_w)*x_fit + (b_w+db_w),
                 color=ORANGE, alpha=0.12, label='Uncertainty band (±σ)')
ann = (f'µ = {mu:.4f} ± {mu_err:.4f} cm⁻¹\n'
       f'N₀ = {N0:.1f} ± {N0_err:.1f}\n'
       f'R² = {r_value**2:.6f}\n'
       f'χ²_red = {chi2_red:.3f}   (dof = {dof})')
ax1.text(0.98, 0.96, ann, transform=ax1.transAxes, fontsize=9.5, va='top', ha='right',
         color=WHITE, fontfamily='monospace',
         bbox=dict(boxstyle='round,pad=0.6', facecolor='#21262d', edgecolor=ORANGE, alpha=0.92))
ax1.set_xlabel('Absorber Thickness  x  (cm)', fontsize=10)
ax1.set_ylabel('ln(N)  —  Natural log of counts', fontsize=10)
style_ax(ax1, 'Linearised Absorption: ln(N) vs x  |  Best Fit by Weighted Least Squares')
leg = ax1.legend(fontsize=9, facecolor=PANEL, edgecolor=BORDER)
for t in leg.get_texts(): t.set_color(WHITE)

# Panel 2: Exponential N vs x
ax2 = fig.add_subplot(gs[1, 0])
ax2.errorbar(df['thickness_mm'], df['counts'], yerr=df['count_error'],
             fmt='s', color=BLUE, ecolor=GREEN, capsize=3, markersize=6,
             zorder=5, label='Observed counts')
ax2.plot(x_fit*10, N_fit, color=ORANGE, lw=2, label='N₀·exp(−µx)', zorder=4)
ax2.set_xlabel('Absorber Thickness  (mm)', fontsize=9)
ax2.set_ylabel('Count rate  N', fontsize=9)
style_ax(ax2, 'Exponential Attenuation  N(x)')
leg2 = ax2.legend(fontsize=8.5, facecolor=PANEL, edgecolor=BORDER)
for t in leg2.get_texts(): t.set_color(WHITE)

# Panel 3: Residuals
ax3 = fig.add_subplot(gs[1, 1])
ax3.errorbar(x, residuals, yerr=yerr, fmt='D', color=PURPLE, ecolor=GREEN,
             capsize=3, markersize=6, zorder=5, label='Residuals Δln(N)')
ax3.axhline(0, color=ORANGE, lw=1.5, linestyle='--', label='Zero line')
ax3.fill_between(x, -yerr, yerr, color=ORANGE, alpha=0.08, label='±1σ envelope')
ax3.set_xlabel('Absorber Thickness  x  (cm)', fontsize=9)
ax3.set_ylabel('Residual  Δln(N)', fontsize=9)
style_ax(ax3, 'Fit Residuals  (random scatter → good fit)')
leg3 = ax3.legend(fontsize=8.5, facecolor=PANEL, edgecolor=BORDER)
for t in leg3.get_texts(): t.set_color(WHITE)

fig.suptitle('Experiment 3 — Linear Absorption Coefficient (µ) via GM Counter',
             fontsize=14, fontweight='bold', color=WHITE, y=0.975)
fig.text(0.5, 0.003,
         'Tri-Chandra Multiple Campus · Department of Physics · B.Sc. III Year · 2082',
         ha='center', fontsize=8.5, color=MUTED)

plt.savefig('practical3_figure.png', dpi=180, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

---
## 6. Discussion & Interpretation

| Quantity | Value |
|---|---|
| Linear attenuation coefficient µ | **1.5392 ± 0.0327 cm⁻¹** |
| Initial count rate N₀ | **1042.5 ± 23.5** |
| R² | **0.999610** |
| Reduced chi-squared χ²_red | **0.094** |

### Key observations

1. **Excellent linearity**: R² = 0.9996 confirms that ln(N) vs x is highly linear, validating the exponential attenuation model.

2. **χ²_red < 1**: A reduced chi-squared of ~0.09 suggests the actual scatter is smaller than the Poisson prediction — the data is very consistent. (Could indicate slight overestimation of errors, or very well-controlled conditions.)

3. **Residuals**: The residuals are small (< 0.06) and randomly distributed around zero — no systematic trend — confirming the model is appropriate.

4. **Physical significance**: The mean free path (average penetration depth before absorption) is:
$$\lambda = \frac{1}{\mu} = \frac{1}{1.5392} \approx 0.65 \text{ cm}$$

### Error analysis
- Statistical error from Poisson fluctuations: propagated via σ_lnN = 1/√N
- Systematic sources not accounted for: dead time of GM tube, background radiation, geometry effects

---
*Author: Nabin Pandey | nabin.795401@trc.tu.edu.np | LinkedIn: nabinpandey1661*